In [ ]:
# !pip install anomalib
# !pip install matplotlib
# !pip install numpy
# !pip install tensorboard
# !pip install torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu128
# !pip install kaggle

In [ ]:
# 2. Импорт необходимых модулей
import os
import copy
import re
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import json
import random
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.optim as optim
import torch.nn.functional as torch_f
import torch.nn as nn

from anomalib.models.image.fastflow.loss import FastflowLoss
from anomalib.models.image.fastflow.torch_model import FastflowModel

from sklearn.metrics import confusion_matrix
import cv2
from skimage import data, filters, measure
from skimage.segmentation import clear_border
# import warnings
# warnings.filterwarnings('ignore')

# Сборка датасетов

Для предобработки картинок и постобработки карт аномалий

In [ ]:
dataset_path = "datasets/processed_printer_dataset"
image_size = 320

# Cap tiles per date to reduce near-duplicate dominance from mass printing.
# Set to None to train/validate on the full date split.
max_train_images_per_date = 300
max_val_images_per_date = 150
num_train_epochs = 6


In [ ]:
def get_augmented_transformer(image_size):
    return transforms.Compose([
        transforms.RandomResizedCrop((image_size, image_size), scale=(0.3, 2.), ratio=(0.3, 2.)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=90),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.RandomApply(torch.nn.ModuleList([
            transforms.GaussianBlur(kernel_size=3, sigma=(0.5, 1.3)),
        ]), p=0.8),
        # transforms.GaussianBlur(kernel_size=5, sigma=(0.05, 1.2)),
        transforms.ToTensor(),
        transforms.RandomErasing(
            p=0.4,
            scale=(0.02, 0.4),
            ratio=(0.1, 5.),
            value=0,
            inplace=False
        ),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

def get_transformer(image_size):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

def show_tensor(tensor):
    pil_img = transforms.ToPILImage()
    pil_img(tensor).show()


def postprocess_anomaly_map(anomaly_map, true_img_size):
    ndims = anomaly_map.ndim
    while anomaly_map.ndim < 4:
        anomaly_map = anomaly_map.unsqueeze(0)

    upsampled_mask = torch_f.interpolate(
        anomaly_map,
        size=true_img_size,
        mode='bilinear',
        align_corners=False
    )
    
    return upsampled_mask.squeeze()

In [ ]:
# path = os.path.join(dataset_path, class_name, "test", test_groups[0], "003.png")
# print(path)
# img = Image.open(path).convert('RGB')
# img = get_transformer(image_size)(img)
# print(img.shape)
# # img = img[0, :, :]
# img = postprocess_anomaly_map(img.unsqueeze(0), true_img_size=(900, 900))
# print(img.shape)
# print(img.max())
# print(img.dtype)
# show_tensor(img)

Датасеты

In [ ]:
class Train_Dataset(Dataset):
    def __init__(self, root_dir, anomalies_dir="anomalies", transform=None, include_dates=None, exclude_dates=None, max_images_per_date=None, seed=42):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.dates = []
        include_dates = set(include_dates) if include_dates is not None else None
        exclude_dates = set(exclude_dates) if exclude_dates is not None else set()
        
        for date_dir in sorted(os.listdir(self.root_dir)):
            if date_dir == anomalies_dir:
                continue
            if include_dates is not None and date_dir not in include_dates:
                continue
            if date_dir in exclude_dates:
                continue

            objects_dir = os.path.join(self.root_dir, date_dir, "objects_parts")
            if os.path.isdir(objects_dir):
                date_image_paths = []
                for img_name in sorted(os.listdir(objects_dir)):
                    if img_name.lower().endswith((".png", ".jpg", ".jpeg")):
                        date_image_paths.append(os.path.join(objects_dir, img_name))

                if max_images_per_date is not None and len(date_image_paths) > max_images_per_date:
                    rng = random.Random(f"{seed}_{date_dir}")
                    date_image_paths = sorted(rng.sample(date_image_paths, max_images_per_date))

                self.image_paths.extend(date_image_paths)
                self.dates.extend([date_dir] * len(date_image_paths))
        
        print(f"Loaded {len(self.image_paths)} object fragments from {self.root_dir}")
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image


def get_date_split(root_dir, validation_fraction=0.2):
    dates = []
    for date_dir in sorted(os.listdir(root_dir)):
        objects_dir = os.path.join(root_dir, date_dir, "objects_parts")
        if os.path.isdir(objects_dir):
            dates.append(date_dir)

    val_count = max(1, int(round(len(dates) * validation_fraction)))
    val_dates = dates[-val_count:]
    train_dates = dates[:-val_count]
    return train_dates, val_dates


def anomaly_index_from_name(name):
    match = re.fullmatch(r"anomaly_(\d+)\.(png|jpg|jpeg)", name.lower())
    return int(match.group(1)) if match else None


def is_good_visual_image(name):
    return re.fullmatch(r"good_\d+\.(png|jpg|jpeg)", name.lower()) is not None


def is_visual_test_anomaly(name, max_index=16):
    anomaly_index = anomaly_index_from_name(name)
    return anomaly_index is not None and anomaly_index <= max_index


def is_head_finetune_anomaly(name, min_index=17):
    anomaly_index = anomaly_index_from_name(name)
    return anomaly_index is not None and anomaly_index >= min_index


def is_visual_test_image(name):
    return is_good_visual_image(name) or is_visual_test_anomaly(name)
    

class Validation_Dataset(Dataset):
    def __init__(self, root_dir, transform=None, return_names=False, filename_filter=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.return_names = return_names
        for img_name in sorted(os.listdir(root_dir)):
            if not img_name.lower().endswith((".png", ".jpg", ".jpeg")):
                continue
            if filename_filter is not None and not filename_filter(img_name):
                continue
            self.image_paths.append(os.path.join(root_dir, img_name))
        
        print(f"Loaded {len(self.image_paths)} object fragments from {self.root_dir}")
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        if self.return_names:
            return image, os.path.basename(self.image_paths[idx])
        else:
            return image


In [ ]:
train_root = os.path.join(dataset_path, "training")
train_dates, val_dates = get_date_split(train_root, validation_fraction=0.2)
print(f"Train dates: {train_dates}")
print(f"Validation dates: {val_dates}")

train_dataset = Train_Dataset(
    train_root,
    transform=get_augmented_transformer(image_size),
    include_dates=train_dates,
    max_images_per_date=max_train_images_per_date,
    seed=random_seed,
)

normal_val_dataset = Train_Dataset(
    train_root,
    transform=get_transformer(image_size),
    include_dates=val_dates,
    max_images_per_date=max_val_images_per_date,
    seed=random_seed,
)

visual_test_dir = os.path.join(dataset_path, "visual_test_images")
visualization_dataset = Validation_Dataset(
    visual_test_dir,
    get_transformer(image_size),
    return_names=True,
    filename_filter=is_visual_test_image,
)
print(visualization_dataset[0][0].shape, visualization_dataset[0][1])
print(f"Train images: {len(train_dataset)}")
print(f"Normal validation images: {len(normal_val_dataset)}")
print(f"Visual test images: {len(visualization_dataset)}")


In [ ]:
from PIL import Image
import torch
import numpy as np

def tensor_to_image(tensor):
    tensor = tensor.cpu().detach()
    denormalize = transforms.Compose([
        transforms.Normalize(mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225], std=[1/0.229, 1/0.224, 1/0.225])
    ])
    tensor = denormalize(tensor)
    if tensor.dim() == 3:
        if tensor.size(0) == 1:
            tensor = tensor.squeeze(0)
        else:
            tensor = tensor.permute(1, 2, 0)
    
    img_np = tensor.numpy()

    if img_np.max() <= 1.0:
        img_np = (img_np * 255).astype(np.uint8)
    else:
        img_np = img_np.astype(np.uint8)

    if len(img_np.shape) == 2:
        return Image.fromarray(img_np, mode='L')
    else:
        return Image.fromarray(img_np)
    
print(train_dataset[0].shape)
tensor_to_image(train_dataset[0]).save("test_img.png")

In [ ]:
def test_images_quality(images_count=100, random_seed=42):
    os.makedirs("./test_normal_images", exist_ok=True)
    print("used existed folder")
    random.seed(random_seed)
    for i, tensor in enumerate(random.choices(train_dataset, k=images_count)):
        tensor_to_image(tensor).save(f"./test_normal_images/img_{i}.png")
test_images_quality()

# Метрики качества

In [ ]:
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score, roc_auc_score
import cv2
import matplotlib.pyplot as plt
from skimage import measure

def img2mask(img):
    mask = img if img.ndim <= 2 else img.squeeze()
    if mask.ndim == 3:
        mask = mask[0, :, :]
    if mask.ndim != 2:
        print("Wrong dimensions")
        raise ValueError
    return mask

def pixel_avg_precision(pred_anomaly_map, true_mask):
    pred_anomaly_map = img2mask(pred_anomaly_map)
    true_mask = img2mask(true_mask)

    y_true = (true_mask.flatten().cpu().numpy() > 0.5).astype(int)
    y_scores = pred_anomaly_map.flatten().cpu().numpy()

    if y_scores.max() > 1.0 or y_scores.min() < 0.0:
        print(f"predicted masks scores are not from [0, 1]")
        raise ValueError

    ap = average_precision_score(y_true, y_scores)
    return ap if not np.isnan(ap) else 0.0


def pixel_roc_auc(pred_anomaly_map, true_mask):
    pred_anomaly_map = img2mask(pred_anomaly_map)
    true_mask = img2mask(true_mask)

    y_true = (true_mask.flatten().cpu().numpy() > 0.5).astype(int)
    y_scores = pred_anomaly_map.flatten().cpu().numpy()

    if y_scores.max() > 1.0 or y_scores.min() < 0.0:
        print(f"predicted masks scores are not from [0, 1]")
        raise ValueError

    auc = roc_auc_score(y_true, y_scores)
    return auc if not np.isnan(auc) else 0.5


def predict_label(pred_anomaly_map, k=0.001):
    """Возвращает скор картинки по среднему k наибольших пикселей"""
    pred_anomaly_map = img2mask(pred_anomaly_map)

    y_scores = pred_anomaly_map.flatten().cpu()
    top_k_scores = torch.topk(y_scores, int(k * len(y_scores)))

    return torch.mean(top_k_scores.values)


def plot_training_curves(metrics: list, metric_names: list, log_dir: str, plot_name: str, log_y=False):
    os.makedirs(log_dir, exist_ok=True)

    plt.figure(figsize=(10, 6))
    for metric, metric_name in zip(metrics, metric_names):
        plt.plot(metric, label=metric_name, marker='o')
    plt.xlabel('Epoch')
    plt.ylabel('Metric')
    if log_y:
        plt.yscale('log')
    plt.title(f'Training and Validation {metric_name}')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(log_dir, plot_name))
    plt.show()


def predict_anomaly_map(model, img, true_img_size=None, augmentations=False):
    aug_anomaly_maps = []
    
    with torch.no_grad():
        anomaly_map = model(img).anomaly_map + 1.
        aug_anomaly_maps.append(anomaly_map)
    
    if augmentations:
        with torch.no_grad():
            img_flip_h = torch.flip(img, dims=[-1])
            anomaly_map_flip_h = model(img_flip_h).anomaly_map + 1.
            anomaly_map_flip_h = torch.flip(anomaly_map_flip_h, dims=[-1])
            aug_anomaly_maps.append(anomaly_map_flip_h)
        
        with torch.no_grad():
            img_flip_v = torch.flip(img, dims=[-2])
            anomaly_map_flip_v = model(img_flip_v).anomaly_map + 1.
            anomaly_map_flip_v = torch.flip(anomaly_map_flip_v, dims=[-2])
            aug_anomaly_maps.append(anomaly_map_flip_v)
    
    avg_anomaly_map = torch.mean(torch.stack(aug_anomaly_maps), dim=0)
    
    if true_img_size is not None:
        avg_anomaly_map = postprocess_anomaly_map(avg_anomaly_map, true_img_size=true_img_size)
    
    return avg_anomaly_map


def visualize_results(original_image, anomaly_map, image_size, true_mask=None, save_path=None, score=None, figsize=(15, 5), alpha=0.7, cmap='jet'):
    denormalize = transforms.Compose([
        transforms.Normalize(mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225], std=[1/0.229, 1/0.224, 1/0.225])
    ])
    original_image = denormalize(original_image)
    original_image = postprocess_anomaly_map(original_image, true_img_size=(image_size, image_size))
    original_image = original_image if original_image.ndim <= 3 else original_image.squeeze()
    anomaly_map = anomaly_map if anomaly_map.ndim <= 3 else anomaly_map.squeeze()

    to_pil = transforms.ToPILImage()
    if isinstance(original_image, torch.Tensor):
        original_image = to_pil(original_image)

    fig, axes = plt.subplots(1, 3, figsize=figsize)
    
    # Оригинальное изображение
    axes[0].imshow(original_image)
    axes[0].set_title('Original image', fontsize=14)
    axes[0].axis('off')
    
    # Маска
    if not (true_mask is None):
        true_mask = true_mask if true_mask.ndim <= 3 else true_mask.squeeze()
        true_mask = img2mask(true_mask)
        if isinstance(true_mask, torch.Tensor):
            true_mask = true_mask.cpu().numpy()
        axes[1].imshow(original_image)
        heatmap = axes[1].imshow(true_mask, cmap=cmap, alpha=alpha, vmin=0., vmax=1.)
        axes[1].set_title('Input + True Mask', fontsize=14)
        axes[1].axis('off')
        plt.colorbar(heatmap, ax=axes[1], fraction=0.046, pad=0.04)


    # Карта аномалий
    anomaly_map = img2mask(anomaly_map)
    if isinstance(anomaly_map, torch.Tensor):
        anomaly_map = anomaly_map.squeeze().cpu().numpy()

    axes[2].imshow(original_image)
    heatmap = axes[2].imshow(anomaly_map, cmap=cmap, alpha=alpha, vmin=0., vmax=1.)
    axes[2].set_title('Input + Anomaly Map', fontsize=14)
    axes[2].axis('off')
    plt.colorbar(heatmap, ax=axes[2], fraction=0.046, pad=0.04)
    
    plt.tight_layout()
    if save_path is None:
        plt.show()
    else:
        plt.savefig(save_path)
        plt.close()

# Обучение модели

In [ ]:
device="cuda" if torch.cuda.is_available() else "cpu"
print(device)
print(f"Название GPU: {torch.cuda.get_device_name(torch.cuda.current_device())}")

In [ ]:
def freeze_feature_extractor(model):
    """Замораживает веса предобученного экстрактора фичей"""
    frozen_params_num = 0
    if hasattr(model, 'feature_extractor'):
        for param in model.feature_extractor.parameters():
            param.requires_grad = False
            frozen_params_num += param.numel()
    print(f"Parameters frozen: {frozen_params_num}")

def get_trainable_parameters(model):
    """Возвращает список обучаемых параметров"""
    trainable_params = []
    train_params_num = 0
    for name, param in model.named_parameters():
        if param.requires_grad:
            trainable_params.append(param)
            train_params_num += param.numel()
    print(f"Trainable parameters: {train_params_num}")
    return trainable_params

In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam, AdamW
from torch.utils.data import DataLoader
from tqdm import tqdm
import os
import copy
import numpy as np
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

def train_fastflow_with_validation(
    model,
    train_dataset,
    validation_dataset,
    visualization_dataset=None,
    num_epochs=10,
    learning_rate=1e-3,
    weight_decay=1e-3,
    eta_min=1e-6,
    device=device,
    log_interval=1,
    log_dir="../experiments",
    freeze_extractor=True,
    batch_size=8,
    num_workers=0,
    resume_from_checkpoint=False,
    early_stopping_patience=5,
    early_stopping_metric="Val_Loss",
    early_stopping_mode="min"
):
    model = model.to(device)
    criterion = FastflowLoss()
    os.makedirs(log_dir, exist_ok=True)
    checkpoints_dir = os.path.join(log_dir, "middle_logs")
    os.makedirs(checkpoints_dir, exist_ok=True)

    print(f"Training samples: {len(train_dataset)}, Validation samples: {len(validation_dataset)}")

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters number: {total_params}")
    if freeze_extractor:
        freeze_feature_extractor(model)
    trainable_params = get_trainable_parameters(model)
    optimizer = optim.AdamW(trainable_params, lr=learning_rate, weight_decay=weight_decay)

    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers
    )
    
    val_loader = DataLoader(
        dataset=validation_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    # scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=3 * len(train_loader), T_mult=2, eta_min=1e-11)
    total_steps = num_epochs * len(train_loader)
    warmup_steps = int(0.1 * total_steps)
    warmup_scheduler = LinearLR(
        optimizer,
        start_factor=0.05,
        end_factor=1.0,
        total_iters=warmup_steps
    )
    main_scheduler = CosineAnnealingLR(
        optimizer,
        T_max=max(1, total_steps - warmup_steps),
        eta_min=eta_min
    )
    scheduler = SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, main_scheduler],
        milestones=[warmup_steps]
    )

    # Early stopping
    best_score = -float('inf') if early_stopping_mode == "max" else float('inf')
    epochs_without_improvement = 0
    best_model_state = None

    checkpoint_path = os.path.join(checkpoints_dir, "checkpoint.pth")
    
    start_epoch = 0
    epoch_scores = {"Train_Loss": [], "Val_Loss": []}
    if resume_from_checkpoint and os.path.exists(checkpoint_path):
        print(f"Loading checkpoint from {checkpoint_path}...")
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        if "scheduler_state_dict" in checkpoint:
            scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        start_epoch = checkpoint["epoch"]
        epoch_scores = checkpoint["epoch_scores"]
        print(f"Resuming from epoch {start_epoch + 1}")
    else:
        print("Starting training from scratch...")

    for epoch in range(start_epoch, num_epochs):
        model.train()
        total_train_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Train Epoch {epoch+1}/{num_epochs}", leave=False):
            if isinstance(batch, (list, tuple)):
                images = batch[0]
            else:
                images = batch
            images = images.to(device)

            optimizer.zero_grad()
            with torch.set_grad_enabled(True):
                outputs = model(images)
                loss = criterion(outputs[0], outputs[1])
                
                loss.backward()
                optimizer.step()
                scheduler.step()
            
            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)
        epoch_scores["Train_Loss"].append(avg_train_loss)

        total_val_loss = 0.0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Val Epoch {epoch+1}/{num_epochs}", leave=False):
                if isinstance(batch, (list, tuple)):
                    images = batch[0]
                else:
                    images = batch
                images = images.to(device)

                outputs = model(images)
                loss = criterion(outputs[0], outputs[1])
                total_val_loss += loss.item()

        avg_val_loss = total_val_loss / len(val_loader)
        epoch_scores["Val_Loss"].append(avg_val_loss)

        if (epoch + 1) % log_interval == 0:
            print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")

            # Визуальная валидация
            if not (visualization_dataset is None):
                model.eval()
                visualization_dir = os.path.join(checkpoints_dir, "visualization", f"epoch_{epoch}")
                os.makedirs(visualization_dir, exist_ok=True)
                with torch.no_grad():
                    for i in range(0, len(visualization_dataset)):
                        img, name = visualization_dataset[i]
                        shapes = img.shape
                        img = img.unsqueeze(0).to(device)
                        
                        anomaly_map = postprocess_anomaly_map(
                            predict_anomaly_map(model, img),
                            true_img_size=(image_size, image_size)
                        )
                        visualize_results(img, anomaly_map, image_size, save_path=os.path.join(visualization_dir, name))

        # Сохранение чекпоинта
        checkpoint_payload = {
            "epoch": epoch + 1,
            "model_state_dict": copy.deepcopy(model.state_dict()),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "epoch_scores": epoch_scores,
            "learning_rate": learning_rate,
        }
        torch.save(checkpoint_payload, checkpoint_path)
        torch.save(checkpoint_payload, os.path.join(checkpoints_dir, f"checkpoint_epoch_{epoch + 1}.pth"))

        checkpoint_metrics_path = os.path.join(checkpoints_dir, "checkpoints_metrics.txt")
        if not os.path.exists(checkpoint_metrics_path) or (not resume_from_checkpoint and epoch == 0):
            with open(checkpoint_metrics_path, "w") as f:
                f.write(f"Epoch {epoch + 1}:\n")
                f.write(f"Train_Loss: {avg_train_loss:.6f}\n")
                f.write(f"Val_Loss: {avg_val_loss:.6f}\n")
                f.write("\n")
        else:
            with open(checkpoint_metrics_path, "a") as f:
                f.write(f"Epoch {epoch + 1}:\n")
                f.write(f"Train_Loss: {avg_train_loss:.6f}\n")
                f.write(f"Val_Loss: {avg_val_loss:.6f}\n")
                f.write("\n")
        
        should_stop = False
        if early_stopping_metric in epoch_scores:
            current_score = epoch_scores[early_stopping_metric][-1]

            is_better = (
                (early_stopping_mode == "max" and current_score > best_score) or
                (early_stopping_mode == "min" and current_score < best_score)
            )

            if is_better:
                best_score = current_score
                epochs_without_improvement = 0
                best_model_state = copy.deepcopy(model.state_dict())
                print(f"New best {early_stopping_metric}: {best_score:.6f}")
            else:
                epochs_without_improvement += 1
                print(f"No improvement for {epochs_without_improvement}/{early_stopping_patience} epochs")

            if epochs_without_improvement >= early_stopping_patience:
                print(f"Early stopping triggered at epoch {epoch + 1}")
                should_stop = True
            
        if should_stop:
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"Loaded best model (score: {best_score:.6f})")
    else:
        print("No early stopping metric found — using final model")

    torch.save(model.state_dict(), os.path.join(log_dir, "trained_model_weights.pth"))
    torch.save(model, os.path.join(log_dir, "trained_model.pth"))
    print(f"Model saved to {log_dir}")

    with open(os.path.join(log_dir, "final_metrics.txt"), "w") as f:
        f.write(f"Train_Loss: {epoch_scores['Train_Loss'][-1]:.6f}\n")
        f.write(f"Val_Loss: {epoch_scores['Val_Loss'][-1]:.6f}\n")
        f.write(f"Best_Val_Loss: {best_score:.6f}\n")

    plot_training_curves([epoch_scores["Train_Loss"], epoch_scores["Val_Loss"]], 
                        ["Train_Loss", "Val_Loss"], 
                        log_dir, "train_val_loss.png")

    return model

In [ ]:
torch.cuda.empty_cache()

In [ ]:
model = FastflowModel(
        backbone="resnet18",
        flow_steps=12,
        input_size=[image_size, image_size],
        pre_trained=True
    )

logs_folder = "3D_printer_resnet_18"
try_number = 7
# Тренировка с валидацией
trained_model = train_fastflow_with_validation(
    model=model,
    train_dataset=train_dataset,
    validation_dataset=normal_val_dataset,
    visualization_dataset=visualization_dataset,
    num_epochs=num_train_epochs,
    learning_rate=3e-5,
    weight_decay=1e-3,
    eta_min=1e-7,
    device=device,
    log_dir=os.path.join("../experiments", logs_folder, f"try_{try_number}"),
    early_stopping_patience=3,
    early_stopping_metric="Val_Loss",
    early_stopping_mode="min",
    batch_size=32
)

In [ ]:
torch.cuda.empty_cache()

Подбор гиперпараметров через optuna с Pixel ROC AUC в качестве метрики качества

In [ ]:
# import optuna
# from optuna.trial import TrialState
# import torch
# import os
# from torch.utils.data import DataLoader

# logs_path = "optuna"
# class_name = "3D_printer"

# def objective(trial):
#     # --- Путь для логов (уникальный для каждого trial) ---
#     os.makedirs("../experiments", exist_ok=True)
#     lg_dir = f"trial_{trial.number}"
#     log_dir = os.path.join("../experiments", class_name)
#     os.makedirs(log_dir, exist_ok=True)
#     log_dir = os.path.join(log_dir, logs_path)
#     os.makedirs(log_dir, exist_ok=True)
#     log_dir = os.path.join(log_dir, lg_dir)
#     os.makedirs(log_dir, exist_ok=True)

#     # --- Гиперпараметры для оптимизации ---
#     flow_steps_num = trial.suggest_int("flow_steps", 3, 5)
#     lr = trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True)
#     weight_decay = trial.suggest_float("weight_decay", 1e-7, 1e-2, log=True)
#     batch_size = trial.suggest_int("batch_size", 14, 14)
#     backbone = "deit_base_distilled_patch16_384"
#     # augmentation_flag = trial.suggest_categorical("augmentation", [False, True])
#     trained_img_size = image_size

#     # train_dataset = Train_Dataset(dataset_path, class_name, get_augmented_transformer(trained_img_size))
#     train_dataset = Train_Dataset(dataset_path, class_name, get_transformer(trained_img_size))
    
#     # Сохранение параметров обучения
#     params_path = os.path.join(log_dir, "params.txt")
#     with open(params_path, "w") as f:
#         f.write(f"flow_steps_num: {flow_steps_num}\n")
#         f.write(f"learning_rate: {lr}\n")
#         f.write(f"weight_decay: {weight_decay}\n")
#         # f.write(f"backbone: {backbone}\n")
#         f.write(f"batch_size: {batch_size}\n")
#         # f.write(f"image_size: {trained_img_size}")

#     # --- Создание модели ---
#     model = FastflowModel(
#         backbone=backbone,
#         flow_steps=flow_steps_num,
#         input_size=[trained_img_size, trained_img_size],
#         pre_trained=True
#     )

#     # --- Обучение модели ---
#     try:
#         trained_model = train_fastflow_with_validation(
#             model=model,
#             train_dataset=train_dataset,
#             num_epochs=20,
#             learning_rate=lr,
#             weight_decay=weight_decay,
#             device=device,
#             log_interval=4,
#             log_dir=log_dir,
#             batch_size=batch_size,
#             resume_from_checkpoint=False,
#             freeze_extractor=True,
#             early_stopping_patience=6,
#             early_stopping_metric="Pixels_ROC_AUC",
#             early_stopping_mode="max"
#         )
#     except Exception as e:
#         print(f"Trial {trial.number} failed with error: {e}")
#         raise optuna.TrialPruned()

#     # Загрузка метрик из файла или из возвращаемого значения

#     metrics_path = os.path.join(log_dir, "final_metrics.txt")
#     if not os.path.exists(metrics_path):
#         raise optuna.TrialPruned()

#     with open(metrics_path, "r") as f:
#         lines = f.readlines()
#         metrics = {}
#         for line in lines:
#             if ":" in line:
#                 key, val = line.strip().split(":")
#                 metrics[key.strip()] = float(val.strip())

#     # Целевая метрика для оптимизации
#     target_metric = "Pixels_ROC_AUC"
#     if target_metric not in metrics:
#         raise optuna.TrialPruned()
    
#     necessary_metric = "Classification_ROC_AUC"
#     if necessary_metric not in metrics:
#         raise optuna.TrialPruned()

#     score = metrics[target_metric] + (metrics[necessary_metric] - 1.)
#     return score


# # Настройка и запуск Optuna
# def run_optuna_optimization(n_trials=50, study_name="fastflow_optimization", storage="sqlite:///fastflow_opt.db"):
#     study = optuna.create_study(
#         study_name=study_name,
#         direction="maximize",
#         storage=storage,
#         load_if_exists=True
#     )

#     study.optimize(objective, n_trials=n_trials, n_jobs=1)

#     # Вывод лучших результатов
#     pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
#     complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

#     print("\nStudy statistics: ")
#     print(f"  Number of finished trials: {len(study.trials)}")
#     print(f"  Number of pruned trials: {len(pruned_trials)}")
#     print(f"  Number of complete trials: {len(complete_trials)}")

#     print("\nBest trial:")
#     trial = study.best_trial
#     print(f"  Value: {trial.value}")
#     print("  Params: ")
#     for key, value in trial.params.items():
#         print(f"    {key}: {value}")
#     print("\n")


#     # Сохранение параметров обучения
#     best_trial_folder = os.path.join("../experiments", class_name, logs_path, "best_trial")
#     os.makedirs(best_trial_folder, exist_ok=True)
#     best_trial_path = os.path.join(best_trial_folder, "best_params.txt")
#     with open(best_trial_path, "w") as f:
#         f.write(f"Value: {trial.value}\n")
#         f.write(f"Trial number: {trial.number}\n")
#         for key, value in trial.params.items():
#             f.write(f"{key}: {value}\n")

#     return study

In [ ]:
# study = run_optuna_optimization(n_trials=30)

# Тестирование лучшей обученной модели

In [ ]:
device="cuda" if torch.cuda.is_available() else "cpu"
print(device)
print(f"Название GPU: {torch.cuda.get_device_name(torch.cuda.current_device())}")

In [ ]:
visual_test_dir = os.path.join(dataset_path, "visual_test_images")
anomalies_dataset = Validation_Dataset(
    visual_test_dir,
    get_transformer(image_size),
    return_names=True,
    filename_filter=is_visual_test_image,
)
class_name = os.path.join("3D_printer_resnet_18", "try_6")
print(f"Visual test images: {len(anomalies_dataset)}")


In [ ]:
print(f"Train images: {len(train_dataset)}")
print(f"Normal validation images: {len(normal_val_dataset)}")


In [ ]:
print(anomalies_dataset[0][0].shape)
tensor_to_image(anomalies_dataset[10][0]).save("test_img.png")

In [ ]:
# Загрузка лучшей модели
# best_model_path = os.path.join("../experiments", class_name, "trained_model.pth")
best_model_path = os.path.join("../experiments", logs_folder, f"try_{try_number}", "trained_model.pth")
best_trained_model = torch.load(best_model_path, weights_only=False)
best_trained_model.to(device)
best_trained_model.eval()

# Тест сегментации на аномальных данных
with torch.no_grad():
    for i in tqdm(range(0, len(anomalies_dataset)), desc="Anomalous Data Segmentation", leave=False):
        img, name = anomalies_dataset[i]
        shapes = img.shape
        img = img.unsqueeze(0).to(device)
        
        anomaly_map = postprocess_anomaly_map(
            predict_anomaly_map(best_trained_model, img), 
            true_img_size=(image_size, image_size)
        )
        print(name)
        visualize_results(img, anomaly_map, image_size)